In [1]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")

print("Dataset shape:", train.shape)
print("Columns:", train.columns.tolist())
print("Missing values:\n", train.isna().sum())
print("Duplicate rows:", train.duplicated().sum())
print("Number of categories:", train["category"].nunique())

display(train.head())
display(train["category"].value_counts())

Dataset shape: (10003, 2)
Columns: ['text', 'category']
Missing values:
 text        0
category    0
dtype: int64
Duplicate rows: 0
Number of categories: 77


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, Length: 77, dtype: int64

In [2]:
category_counts = train["category"].value_counts()

print("Number of categories:", category_counts.size)
print("Fewest tickets in a category:", category_counts.min())
print("Most tickets in a category:", category_counts.max())
print("Average tickets per category:", round(category_counts.mean(), 1))

display(category_counts.to_frame("ticket_count").head(15))

Number of categories: 77
Fewest tickets in a category: 35
Most tickets in a category: 187
Average tickets per category: 129.9


,ticket_count
category,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
transaction_charged_twice,175
declined_cash_withdrawal,173
transfer_fee_charged,172
transfer_not_received_by_recipient,171


In [3]:
category_counts = train["category"].value_counts()
selected_categories = category_counts.head(15).index.tolist()

train_mvp = train[train["category"].isin(selected_categories)].copy()

print("MVP dataset shape:", train_mvp.shape)
print("Number of categories:", train_mvp["category"].nunique())

display(train_mvp["category"].value_counts().to_frame("ticket_count"))

MVP dataset shape: (2602, 2)
Number of categories: 15


,ticket_count
category,
card_payment_fee_charged,187
direct_debit_payment_not_recognised,182
balance_not_updated_after_cheque_or_cash_deposit,181
wrong_amount_of_cash_received,180
cash_withdrawal_charge,177
transaction_charged_twice,175
declined_cash_withdrawal,173
transfer_fee_charged,172
transfer_not_received_by_recipient,171


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    train_mvp["text"],
    train_mvp["category"],
    test_size=0.2,
    random_state=42,
    stratify=train_mvp["category"]
)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": SVC(kernel="linear", random_state=42),
    "RBF SVM": SVC(kernel="rbf", random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, classifier in classifiers.items():
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=20000
        )),
        ("classifier", classifier)
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "macro_f1": f1_score(y_val, y_pred, average="macro")
    })

results_df = pd.DataFrame(results).sort_values(
    by="macro_f1",
    ascending=False
).reset_index(drop=True)

display(results_df.round(3))

,model,accuracy,macro_f1
0,Linear SVM,0.919,0.918
1,RBF SVM,0.898,0.897
2,Logistic Regression,0.887,0.886
3,Random Forest,0.875,0.873
4,Multinomial Naive Bayes,0.868,0.866
5,K-Nearest Neighbors,0.800,0.799
6,Decision Tree,0.800,0.799


### Model Comparison

Linear SVM achieved the highest validation accuracy (0.919) and macro-F1 score (0.918) among the seven models tested. It showed the best overall balance across the 15 categories. RBF SVM ranked second, with a validation accuracy of 0.898 and a macro-F1 score of 0.897. Therefore, Linear SVM is selected as the best candidate for this ticket classification task. Its final performance will be checked on the untouched test set.

In [6]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

test = pd.read_csv("../data/raw/test.csv")
test_mvp = test[test["category"].isin(selected_categories)].copy()

final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("classifier", SVC(kernel="linear", random_state=42))
])

final_model.fit(train_mvp["text"], train_mvp["category"])
test_predictions = final_model.predict(test_mvp["text"])

print("Test dataset shape:", test_mvp.shape)
print("Number of test categories:", test_mvp["category"].nunique())
print("Test accuracy:", round(accuracy_score(test_mvp["category"], test_predictions), 3))
print("Test macro F1:", round(f1_score(
    test_mvp["category"],
    test_predictions,
    average="macro"
), 3))

report = classification_report(
    test_mvp["category"],
    test_predictions,
    output_dict=True,
    zero_division=0
)

display(pd.DataFrame(report).T.round(3))

Test dataset shape: (600, 2)
Number of test categories: 15
Test accuracy: 0.92
Test macro F1: 0.92


,precision,recall,f1-score,support
balance_not_updated_after_bank_transfer,0.850,0.850,0.850,40.00
balance_not_updated_after_cheque_or_cash_deposit,0.975,0.975,0.975,40.00
card_payment_fee_charged,0.854,0.875,0.864,40.00
card_payment_not_recognised,0.947,0.900,0.923,40.00
card_payment_wrong_exchange_rate,0.881,0.925,0.902,40.00
cash_withdrawal_charge,0.952,1.000,0.976,40.00
declined_cash_withdrawal,0.929,0.975,0.951,40.00
direct_debit_payment_not_recognised,0.946,0.875,0.909,40.00
extra_charge_on_statement,0.854,0.875,0.864,40.00
request_refund,1.000,0.975,0.987,40.00


### Final Model Evaluation

Linear SVM was selected after comparing seven classifiers on the validation set. It achieved the highest validation accuracy (0.919) and macro-F1 score (0.918). On the held-out test set, Linear SVM achieved an accuracy of 0.920 and a macro-F1 score of 0.920 across 600 tickets from 15 categories. The similar validation and test scores show no noticeable performance drop on this split. The model performed best on `transaction_charged_twice` (F1 = 1.000), while `transfer_not_received_by_recipient` had the lowest class F1 score (0.824). Based on these results, Linear SVM is the strongest candidate among the tested models for this 15-category MVP.

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    train_mvp["text"],
    train_mvp["category"],
    test_size=0.2,
    random_state=42,
    stratify=train_mvp["category"]
)

validation_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("classifier", SVC(kernel="linear", random_state=42))
])

validation_model.fit(X_train, y_train)
validation_predictions = validation_model.predict(X_val)

validation_results = pd.DataFrame({
    "text": X_val.to_numpy(),
    "true_category": y_val.to_numpy(),
    "predicted_category": validation_predictions
})

errors = validation_results[
    validation_results["true_category"] != validation_results["predicted_category"]
].copy()

error_pairs = (
    errors.groupby(["true_category", "predicted_category"])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("count")
    .reset_index()
)

print("Number of validation errors:", len(errors))
print("Most common category confusions:")
display(error_pairs)

print("Example misclassified tickets:")
display(errors.head(10))

Number of validation errors: 42
Most common category confusions:


,true_category,predicted_category,count
0,balance_not_updated_after_bank_transfer,transfer_not_received_by_recipient,6
1,transfer_not_received_by_recipient,balance_not_updated_after_bank_transfer,5
2,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate,3
3,direct_debit_payment_not_recognised,card_payment_not_recognised,2
4,extra_charge_on_statement,transfer_fee_charged,2
5,wrong_amount_of_cash_received,declined_cash_withdrawal,2
6,card_payment_wrong_exchange_rate,wrong_exchange_rate_for_cash_withdrawal,1
7,card_payment_not_recognised,extra_charge_on_statement,1
8,card_payment_fee_charged,transfer_fee_charged,1
9,card_payment_not_recognised,card_payment_fee_charged,1


Example misclassified tickets:


,text,true_category,predicted_category
11,Will I get extra charges?,transfer_fee_charged,extra_charge_on_statement
18,You shorted me money when I tried to make a wi...,wrong_amount_of_cash_received,declined_cash_withdrawal
36,I don't appreciate the extra fees being siphon...,transfer_fee_charged,card_payment_fee_charged
47,I think I was charged a different exchange rat...,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate
54,my atm transaction was wrong,wrong_amount_of_cash_received,declined_cash_withdrawal
55,The money has not yet been placed in recipient...,transfer_not_received_by_recipient,balance_not_updated_after_cheque_or_cash_deposit
62,Im going to need to make a transfer from a UK ...,balance_not_updated_after_bank_transfer,transfer_not_received_by_recipient
63,I have a fee and I don't know why,card_payment_fee_charged,extra_charge_on_statement
73,i accidentally entered in my retailers informa...,transfer_not_received_by_recipient,card_payment_not_recognised
74,Is there an allowance on cards?,card_payment_fee_charged,card_payment_not_recognised


### Validation Error Analysis

The Linear SVM made 42 errors on the validation set. The most frequent confusion was between `balance_not_updated_after_bank_transfer` and `transfer_not_received_by_recipient`, with 11 errors across both directions. These intents are related: customers may describe a transfer that has not arrived or a balance that has not updated.

The model also confused some fee-related and exchange-rate categories. Short or vague tickets can make the customer's intent difficult to identify from text alone. A confidence-based review step could send uncertain predictions to a human agent.

In [8]:
from sklearn.calibration import CalibratedClassifierCV

base_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("classifier", SVC(kernel="linear", random_state=42))
])

calibrated_model = CalibratedClassifierCV(
    estimator=base_pipeline,
    method="sigmoid",
    cv=3
)

calibrated_model.fit(X_train, y_train)

val_probabilities = calibrated_model.predict_proba(X_val)
val_predictions = calibrated_model.classes_[val_probabilities.argmax(axis=1)]
val_confidence = val_probabilities.max(axis=1)
y_val_array = y_val.to_numpy()

threshold_results = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    auto_mask = val_confidence >= threshold

    if auto_mask.any():
        auto_accuracy = accuracy_score(
            y_val_array[auto_mask],
            val_predictions[auto_mask]
        )
    else:
        auto_accuracy = None

    threshold_results.append({
        "confidence_threshold": threshold,
        "auto_classified_percent": auto_mask.mean() * 100,
        "manual_review_percent": (~auto_mask).mean() * 100,
        "accuracy_on_auto_classified": auto_accuracy,
        "auto_classified_count": int(auto_mask.sum())
    })

display(pd.DataFrame(threshold_results).round(3))

,confidence_threshold,auto_classified_percent,manual_review_percent,accuracy_on_auto_classified,auto_classified_count
0,0.5,96.545,3.455,0.928,503
1,0.6,86.372,13.628,0.949,450
2,0.7,55.854,44.146,0.969,291
3,0.8,4.415,95.585,0.957,23
4,0.9,0.000,100.000,NaN,0


### Confidence Threshold Analysis

The confidence threshold controls the trade-off between automation coverage and prediction reliability. At a threshold of 0.70, the model automatically classified 55.9% of validation tickets with 96.9% accuracy on those predictions. The remaining 44.1% were flagged for manual review. A threshold of 0.60 increased automation coverage to 86.4%, with 94.9% accuracy on automatically classified tickets. For this MVP, 0.70 is a reasonable starting point. The threshold can be adjusted based on the acceptable balance between prediction reliability and manual-review workload.

In [9]:
final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("classifier", SVC(kernel="linear", random_state=42))
])

final_calibrated_model = CalibratedClassifierCV(
    estimator=final_pipeline,
    method="sigmoid",
    cv=3
)

final_calibrated_model.fit(train_mvp["text"], train_mvp["category"])

test = pd.read_csv("../data/raw/test.csv")
test_mvp = test[test["category"].isin(selected_categories)].copy()

test_probabilities = final_calibrated_model.predict_proba(test_mvp["text"])
test_predictions = final_calibrated_model.classes_[test_probabilities.argmax(axis=1)]
test_confidence = test_probabilities.max(axis=1)

confidence_threshold = 0.70
auto_mask = test_confidence >= confidence_threshold

print("Test tickets:", len(test_mvp))
print("Overall test accuracy:", round(
    accuracy_score(test_mvp["category"], test_predictions), 3
))
print("Overall test macro F1:", round(
    f1_score(test_mvp["category"], test_predictions, average="macro"), 3
))
print("Automatically classified:", int(auto_mask.sum()))
print("Manual review:", int((~auto_mask).sum()))
print("Automatic coverage (%):", round(auto_mask.mean() * 100, 1))
print("Manual review rate (%):", round((~auto_mask).mean() * 100, 1))
print("Accuracy on automatically classified tickets:", round(
    accuracy_score(
        test_mvp["category"].to_numpy()[auto_mask],
        test_predictions[auto_mask]
    ), 3
))

Test tickets: 600
Overall test accuracy: 0.908
Overall test macro F1: 0.908
Automatically classified: 431
Manual review: 169
Automatic coverage (%): 71.8
Manual review rate (%): 28.2
Accuracy on automatically classified tickets: 0.954


### Final Confidence-Based Evaluation

The calibrated Linear SVM achieved an overall test accuracy and macro-F1 score of 0.908 across 600 tickets from 15 categories. With a confidence threshold of 0.70, it automatically classified 431 tickets (71.8%) and flagged 169 tickets (28.2%) for manual review. Accuracy on automatically classified tickets was 0.954. This shows the review policy increased reliability among auto-handled tickets while leaving uncertain cases for a human agent. The validation set showed a different automatic coverage at the same threshold, so coverage should be monitored on new data.

In [10]:
model = calibrated_model

sample_tickets = [
    "My card was swallowed by the ATM.",
    "I transferred money, but the recipient has not received it.",
    "I do not recognize this transaction on my account.",
    "How can I change my PIN?"
]

threshold = 0.70

for ticket in sample_tickets:
    probabilities = model.predict_proba([ticket])[0]
    best_index = probabilities.argmax()
    confidence = probabilities[best_index]
    predicted_category = model.classes_[best_index]

    if confidence >= threshold:
        result = predicted_category
    else:
        result = "Manual review"

    print(f"Ticket: {ticket}")
    print(f"Prediction: {result}")
    print(f"Confidence: {confidence:.3f}")
    print()

Ticket: My card was swallowed by the ATM.
Prediction: declined_cash_withdrawal
Confidence: 0.708

Ticket: I transferred money, but the recipient has not received it.
Prediction: transfer_not_received_by_recipient
Confidence: 0.743

Ticket: I do not recognize this transaction on my account.
Prediction: Manual review
Confidence: 0.631

Ticket: How can I change my PIN?
Prediction: Manual review
Confidence: 0.423



### Custom Prediction Examples

The model correctly identified the transfer issue and sent two low-confidence tickets for manual review. However, it incorrectly classified the swallowed-card ticket as `declined_cash_withdrawal` with a confidence of 0.708. This shows that confidence scores do not guarantee correct predictions, especially for tickets that differ from the training examples. The confidence threshold should be evaluated on representative validation data.

In [11]:
from pathlib import Path
import joblib

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

model_dir = project_dir / "models"
model_dir.mkdir(exist_ok=True)

model_file = model_dir / "support_ticket_classifier.joblib"

joblib.dump(
    {
        "model": model,
        "threshold": threshold
    },
    model_file
)

print(f"Model saved to: {model_file.resolve()}")

Model saved to: C:\Users\alexk\OneDrive\Desktop\ml_projects\support-ticket-classifier\models\support_ticket_classifier.joblib


In [12]:
import joblib

saved_artifact = joblib.load(model_file)
saved_model = saved_artifact["model"]
saved_threshold = saved_artifact["threshold"]

def classify_ticket(text):
    probabilities = saved_model.predict_proba([text])[0]
    best_index = probabilities.argmax()
    confidence = float(probabilities[best_index])

    if confidence >= saved_threshold:
        category = saved_model.classes_[best_index]
    else:
        category = "Manual review"

    return {
        "category": category,
        "confidence": round(confidence, 3)
    }

classify_ticket("I transferred money, but the recipient has not received it.")

{'category': 'transfer_not_received_by_recipient', 'confidence': 0.743}

## Project Conclusion

Seven classification models were compared using the same validation split. Linear SVM achieved the best validation results, with an accuracy of 0.919 and a macro-F1 score of 0.918, so it was selected for the final model.

The calibrated Linear SVM achieved a test accuracy and macro-F1 score of 0.908 across 600 tickets from 15 categories. With a confidence threshold of 0.70, it automatically classified 71.8% of test tickets. Accuracy on those automatically classified tickets was 0.954; the remaining 28.2% were flagged for manual review.

The saved model can be loaded and used to classify new ticket text. This MVP covers 15 selected categories, so it should send tickets outside those categories or with uncertain predictions for manual review.

In [13]:
from pathlib import Path
import pandas as pd

data_dirs = [
    Path("data/raw"),
    Path("../data/raw"),
    Path("dataset_download")
]

data_dir = next(
    (path for path in data_dirs if (path / "train.csv").exists() and (path / "test.csv").exists()),
    None
)

if data_dir is None:
    raise FileNotFoundError("Could not find train.csv and test.csv.")

train_full = pd.read_csv(data_dir / "train.csv")
test_full = pd.read_csv(data_dir / "test.csv")

print(f"Training data shape: {train_full.shape}")
print(f"Test data shape: {test_full.shape}")
print(f"Training categories: {train_full['category'].nunique()}")
print(f"Test categories: {test_full['category'].nunique()}")

Training data shape: (10003, 2)
Test data shape: (3080, 2)
Training categories: 77
Test categories: 77


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

X_train_77, X_valid_77, y_train_77, y_valid_77 = train_test_split(
    train_full["text"],
    train_full["category"],
    test_size=0.2,
    random_state=42,
    stratify=train_full["category"]
)

vectorizer_77 = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)

X_train_vec_77 = vectorizer_77.fit_transform(X_train_77)
X_valid_vec_77 = vectorizer_77.transform(X_valid_77)

classifiers_77 = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": SVC(kernel="linear"),
    "RBF SVM": SVC(kernel="rbf"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    )
}

results_77 = []

for name, classifier in classifiers_77.items():
    print(f"Training {name}...")
    classifier.fit(X_train_vec_77, y_train_77)
    predictions = classifier.predict(X_valid_vec_77)

    results_77.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid_77, predictions),
        "Macro F1": f1_score(y_valid_77, predictions, average="macro")
    })

results_77_df = pd.DataFrame(results_77).sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

results_77_df

Training Logistic Regression...
Training Multinomial Naive Bayes...
Training Linear SVM...
Training RBF SVM...
Training K-Nearest Neighbors...
Training Decision Tree...
Training Random Forest...


,Model,Accuracy,Macro F1
0,Linear SVM,0.870565,0.871086
1,RBF SVM,0.863068,0.863829
2,Logistic Regression,0.844578,0.841855
3,Random Forest,0.834583,0.834465
4,Multinomial Naive Bayes,0.807596,0.780580
5,K-Nearest Neighbors,0.731134,0.728941
6,Decision Tree,0.718641,0.720061


### Model Comparison on 77 Categories

Linear SVM achieved the best validation results, with an accuracy of 0.871 and a macro-F1 score of 0.871. RBF SVM ranked second, with a macro-F1 score of 0.864. Linear SVM was selected for final testing because it performed best on both metrics. The test set was not used during model selection.

In [15]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

final_vectorizer_77 = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)

X_train_full_vec_77 = final_vectorizer_77.fit_transform(train_full["text"])
X_test_full_vec_77 = final_vectorizer_77.transform(test_full["text"])

final_model_77 = SVC(kernel="linear")
final_model_77.fit(X_train_full_vec_77, train_full["category"])

test_predictions_77 = final_model_77.predict(X_test_full_vec_77)

print(f"Test accuracy: {accuracy_score(test_full['category'], test_predictions_77):.3f}")
print(
    f"Test macro-F1: "
    f"{f1_score(test_full['category'], test_predictions_77, average='macro'):.3f}"
)

report_77 = classification_report(
    test_full["category"],
    test_predictions_77,
    output_dict=True,
    zero_division=0
)

report_77_df = pd.DataFrame(report_77).T
report_77_df.loc[final_model_77.classes_].nsmallest(10, "f1-score")[["precision", "recall", "f1-score"]]

Test accuracy: 0.893
Test macro-F1: 0.894


,precision,recall,f1-score
balance_not_updated_after_bank_transfer,0.659091,0.725,0.690476
card_not_working,0.622951,0.950,0.752475
transfer_not_received_by_recipient,0.738095,0.775,0.756098
top_up_failed,0.744186,0.800,0.771084
pending_transfer,0.852941,0.725,0.783784
topping_up_by_card,0.857143,0.750,0.800000
transfer_timing,0.785714,0.825,0.804878
wrong_exchange_rate_for_cash_withdrawal,0.882353,0.750,0.810811
card_arrival,0.790698,0.850,0.819277
card_acceptance,0.842105,0.800,0.820513


### Final Test Evaluation

The Linear SVM achieved a test accuracy of 0.893 and a macro-F1 score of 0.894 across 3,080 tickets from 77 categories. The test scores were slightly higher than the validation scores.

Performance varied across categories. `balance_not_updated_after_bank_transfer` had the lowest F1 score (0.690). For `card_not_working`, recall was high (0.950), while precision was lower (0.623), meaning the model found most tickets in this category but also assigned this label to some tickets from other categories. The most common confusion pairs should be reviewed before finalizing the model.

In [16]:
import numpy as np
from sklearn.metrics import confusion_matrix

labels_77 = final_model_77.classes_

confusion_77 = confusion_matrix(
    test_full["category"],
    test_predictions_77,
    labels=labels_77
)

confusion_off_diagonal = confusion_77.copy()
np.fill_diagonal(confusion_off_diagonal, 0)

confusion_pairs = []

for actual_index, predicted_index in zip(*np.where(confusion_off_diagonal > 0)):
    confusion_pairs.append({
        "Actual category": labels_77[actual_index],
        "Predicted category": labels_77[predicted_index],
        "Count": confusion_off_diagonal[actual_index, predicted_index]
    })

top_confusions_77 = (
    pd.DataFrame(confusion_pairs)
    .sort_values("Count", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

top_confusions_77

,Actual category,Predicted category,Count
0,card_acceptance,card_not_working,5
1,top_up_reverted,top_up_failed,5
2,wrong_exchange_rate_for_cash_withdrawal,card_payment_wrong_exchange_rate,5
3,verify_my_identity,why_verify_identity,5
4,transfer_not_received_by_recipient,balance_not_updated_after_bank_transfer,5
5,balance_not_updated_after_bank_transfer,transfer_not_received_by_recipient,4
6,contactless_not_working,card_not_working,4
7,pending_top_up,top_up_failed,4
8,fiat_currency_support,exchange_via_app,4
9,unable_to_verify_identity,verify_my_identity,4


### Error Analysis

The most common errors occurred between semantically similar categories. Examples include `top_up_reverted` versus `top_up_failed`, and `transfer_not_received_by_recipient` versus `balance_not_updated_after_bank_transfer`. The most frequent confusion pairs appeared 4–5 times each. This suggests that some ticket categories have overlapping wording and are difficult to distinguish.

In [17]:
from pathlib import Path
import joblib

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

model_dir = project_dir / "models"
model_dir.mkdir(exist_ok=True)

final_model_file_77 = model_dir / "support_ticket_classifier_77.joblib"

joblib.dump(
    {
        "vectorizer": final_vectorizer_77,
        "model": final_model_77,
        "categories": list(final_model_77.classes_)
    },
    final_model_file_77
)

print(f"Final 77-category model saved to: {final_model_file_77.resolve()}")

Final 77-category model saved to: C:\Users\alexk\OneDrive\Desktop\ml_projects\support-ticket-classifier\models\support_ticket_classifier_77.joblib


In [18]:
import joblib

saved_artifact_77 = joblib.load(final_model_file_77)

sample_tickets = [
    "I transferred money, but the recipient has not received it.",
    "My card is not working in the ATM."
]

for ticket in sample_tickets:
    ticket_vector = saved_artifact_77["vectorizer"].transform([ticket])
    prediction = saved_artifact_77["model"].predict(ticket_vector)[0]

    print(f"Ticket: {ticket}")
    print(f"Predicted category: {prediction}")
    print()

Ticket: I transferred money, but the recipient has not received it.
Predicted category: transfer_not_received_by_recipient

Ticket: My card is not working in the ATM.
Predicted category: card_swallowed



## Project Summary

Seven classification models were compared on the validation set across all 77 ticket categories. Linear SVM achieved the best results, with an accuracy and macro-F1 score of 0.871, so it was selected for final testing.

On the separate test set of 3,080 tickets, Linear SVM achieved an accuracy of 0.893 and a macro-F1 score of 0.894. The most frequent errors occurred between semantically similar categories, such as `top_up_reverted` and `top_up_failed`.

The final model and TF-IDF vectorizer were saved in `models/support_ticket_classifier_77.joblib` and successfully loaded for prediction. The current model returns one category for every ticket and does not provide a calibrated confidence score or manual-review routing.